# Lekcja 08 - Wzorzec projektowy Multi-Agent


## Konfiguracja


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Dlaczego systemy wieloagentowe?

Zadania ze świata rzeczywistego, takie jak planowanie podróży, wymagają różnych rodzajów wiedzy — logistyki, wiedzy lokalnej, budżetowania i nie tylko. Pojedynczy agent próbujący wszystko ogarnąć, szybko staje się nieporęczny.

Systemy wieloagentowe rozwiązują to dzięki **specjalizacji**: każdy agent koncentruje się na jednej dziedzinie, dostarczając wyniki wyższej jakości niż ogólniak. Poprawiają też **skalowalność** — możesz dodać nowych agentów (np. specjalistę od lotów, krytyka restauracji) bez przepisywania istniejącego przepływu pracy. Agenci współpracują przez ustrukturyzowany pipeline, przekazując kontekst od jednego do drugiego.


## Tworzenie wyspecjalizowanych agentów


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Budowanie kolejnego workflow

`WorkflowBuilder` pozwala na połączenie agentów w graf skierowany. Tutaj tworzymy prosty dwustopniowy pipeline: **TravelPlanner** tworzy plan podróży, a następnie **TravelConcierge** go przegląda i ulepsza.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodawanie większej liczby agentów do przepływu pracy

Jedną z największych zalet wzorca wieloagentowego jest łatwość jego rozszerzania. Poniżej dodajemy agenta **BudgetReviewer**, który sprawdza plan pod kątem budżetu podróżnego, oznacza pozycje, które mogą spowodować przekroczenie limitu kosztów, i sugeruje oszczędne alternatywy. Przepływ pracy teraz uruchamia trzech agentów w kolejności:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Tworzyć wyspecjalizowane agenty** — każdy z określoną rolą (planowanie, konsjerż, przegląd budżetu).
2. **Łączyć agentów w sekwencyjny przepływ pracy** za pomocą `WorkflowBuilder` i `add_edge`.
3. **Strumieniować dane wyjściowe** z potoku wieloagentowego, śledząc, który agent mówi.
4. **Rozszerzać przepływ pracy** dodając nowych agentów do łańcucha bez modyfikowania istniejących.

Wzorzec projektowy wieloagentowy utrzymuje prostotę każdego agenta, jednocześnie generując bogatsze, bardziej szczegółowo zweryfikowane wyniki, niż mógłby osiągnąć pojedynczy agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
